# Day 1: Bayesian Thinking

**Applied Bayesian Statistics Workshop**

---

## Learning Objectives

1. Understand Bayes' theorem through a concrete coin-flip example
2. Work with the Beta-Binomial conjugate model analytically and visually
3. Perform prior predictive checks
4. Specify and fit a simple model in PyMC
5. Build a Bayesian linear regression from scratch
6. Compare credible intervals with confidence intervals

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pymc as pm
import arviz as az

# Plotting defaults
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12
az.style.use("arviz-darkgrid")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 1. Bayes' Theorem Intuition: The Coin Flip

We observe a coin flipped $N$ times, producing $k$ heads. We want to estimate the
probability of heads, $\theta$.

$$
P(\theta \mid \text{data}) = \frac{P(\text{data} \mid \theta)\, P(\theta)}{P(\text{data})}
$$

- **Prior** $P(\theta)$: our belief about $\theta$ before seeing data
- **Likelihood** $P(\text{data} \mid \theta)$: how probable the data are given $\theta$
- **Posterior** $P(\theta \mid \text{data})$: updated belief after observing data

In [ ]:
# --- Discrete grid approximation of Bayes' theorem ---
theta_grid = np.linspace(0, 1, 1000)

# Observed data: 7 heads in 10 flips
N_flips = 10
k_heads = 7

# Prior: Uniform(0, 1) = Beta(1, 1)
prior = stats.beta.pdf(theta_grid, 1, 1)

# Likelihood: Binomial(k | N, theta)
likelihood = stats.binom.pmf(k_heads, N_flips, theta_grid)

# Unnormalized posterior
unnorm_posterior = likelihood * prior

# Normalize (trapezoidal integration)
posterior = unnorm_posterior / np.trapz(unnorm_posterior, theta_grid)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].fill_between(theta_grid, prior, alpha=0.4, color="steelblue")
axes[0].set_title("Prior: Beta(1, 1)")
axes[0].set_xlabel(r"$\theta$")

axes[1].fill_between(theta_grid, likelihood, alpha=0.4, color="coral")
axes[1].set_title(f"Likelihood: {k_heads} heads in {N_flips} flips")
axes[1].set_xlabel(r"$\theta$")

axes[2].fill_between(theta_grid, posterior, alpha=0.4, color="mediumpurple")
axes[2].set_title("Posterior")
axes[2].set_xlabel(r"$\theta$")

for ax in axes:
    ax.set_ylabel("Density")
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.suptitle("Bayes' Theorem: Grid Approximation", y=1.03, fontsize=14)
plt.show()

## 2. Beta-Binomial Conjugate Model

When the prior is $\text{Beta}(\alpha, \beta)$ and the likelihood is $\text{Binomial}(N, \theta)$,
the posterior is available in closed form:

$$
\theta \mid k, N \;\sim\; \text{Beta}(\alpha + k, \; \beta + N - k)
$$

This is called **conjugacy** and it lets us update beliefs without MCMC.

In [ ]:
# --- Analytical Beta-Binomial update ---
alpha_prior, beta_prior = 2, 5  # Slightly skeptical prior: expect heads < 0.5

# Sequential data: observe flips one batch at a time
batches = [(3, 5), (6, 10), (14, 20)]  # (heads, flips)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.linspace(0, 1, 500)

# Plot prior
ax.plot(x, stats.beta.pdf(x, alpha_prior, beta_prior),
        lw=2, ls="--", label=f"Prior: Beta({alpha_prior}, {beta_prior})")

a, b = alpha_prior, beta_prior
cumulative_k, cumulative_n = 0, 0

for heads, flips in batches:
    cumulative_k += heads
    cumulative_n += flips
    a_post = alpha_prior + cumulative_k
    b_post = beta_prior + cumulative_n - cumulative_k
    label = f"After {cumulative_k}/{cumulative_n} heads: Beta({a_post}, {b_post})"
    ax.plot(x, stats.beta.pdf(x, a_post, b_post), lw=2, label=label)

ax.axvline(0.5, color="gray", ls=":", alpha=0.6, label="Fair coin")
ax.set_xlabel(r"$\theta$ (probability of heads)")
ax.set_ylabel("Density")
ax.set_title("Sequential Bayesian Updating with Beta-Binomial Model")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Print posterior summary
print(f"Final posterior: Beta({a_post}, {b_post})")
print(f"Posterior mean: {a_post / (a_post + b_post):.3f}")
print(f"Posterior mode: {(a_post - 1) / (a_post + b_post - 2):.3f}")
print(f"95% credible interval: ({stats.beta.ppf(0.025, a_post, b_post):.3f}, "
      f"{stats.beta.ppf(0.975, a_post, b_post):.3f})")

## 3. Prior Predictive Checks

Before seeing real data, we simulate datasets implied by the prior to ask:
*"Does our prior produce data that look reasonable?"*

$$
y^{\text{prior}} \sim \int P(y \mid \theta)\, P(\theta)\, d\theta
$$

In [ ]:
# --- Prior predictive check for Beta-Binomial model ---
n_simulations = 5000
n_trials = 20

prior_configs = [
    (1, 1, "Uniform: Beta(1,1)"),
    (2, 5, "Skeptical: Beta(2,5)"),
    (10, 10, "Centered: Beta(10,10)"),
    (0.5, 0.5, "Jeffreys: Beta(0.5,0.5)"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (a, b, title) in zip(axes.ravel(), prior_configs):
    # Step 1: sample theta from the prior
    theta_samples = rng.beta(a, b, size=n_simulations)
    # Step 2: for each theta, simulate data
    y_prior_pred = rng.binomial(n_trials, theta_samples)

    ax.hist(y_prior_pred, bins=np.arange(-0.5, n_trials + 1.5),
            density=True, color="steelblue", edgecolor="white", alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel(f"Number of heads (out of {n_trials})")
    ax.set_ylabel("Proportion")
    ax.set_xlim(-0.5, n_trials + 0.5)

plt.suptitle("Prior Predictive Distributions for Different Priors", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Introduction to PyMC

PyMC is a probabilistic programming library. We specify a generative model
and PyMC handles the inference.

Let's start with the same coin-flip problem, now using PyMC.

In [ ]:
# --- Simple coin model in PyMC ---
observed_heads = 7
total_flips = 10

with pm.Model() as coin_model:
    # Prior
    theta = pm.Beta("theta", alpha=2, beta=5)

    # Likelihood
    y = pm.Binomial("y", n=total_flips, p=theta, observed=observed_heads)

    # Sample from the posterior
    trace_coin = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print(az.summary(trace_coin, var_names=["theta"]))

In [ ]:
# --- Visualize the posterior ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ArviZ trace plot
az.plot_trace(trace_coin, var_names=["theta"], axes=axes.reshape(1, 2))

plt.tight_layout()
plt.show()

In [ ]:
# Compare PyMC posterior with the analytical posterior
fig, ax = plt.subplots(figsize=(10, 5))

# Analytical: Beta(2 + 7, 5 + 3) = Beta(9, 8)
x = np.linspace(0, 1, 500)
ax.plot(x, stats.beta.pdf(x, 9, 8), "r-", lw=2.5, label="Analytical: Beta(9, 8)")

# PyMC samples
theta_samples = az.extract(trace_coin, var_names=["theta"])["theta"].values
ax.hist(theta_samples, bins=50, density=True, alpha=0.5,
        color="steelblue", edgecolor="white", label="PyMC samples")

ax.set_xlabel(r"$\theta$")
ax.set_ylabel("Density")
ax.set_title("PyMC vs Analytical Posterior")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Bayesian Linear Regression

Model:
$$
y_i = \alpha + \beta x_i + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)
$$

Priors:
$$
\alpha \sim \mathcal{N}(0, 10), \quad \beta \sim \mathcal{N}(0, 10), \quad \sigma \sim \text{HalfNormal}(5)
$$

In [ ]:
# --- Generate synthetic data ---
true_alpha = 1.5
true_beta = 2.3
true_sigma = 0.8
n_obs = 100

x = rng.uniform(-2, 2, size=n_obs)
y = true_alpha + true_beta * x + rng.normal(0, true_sigma, size=n_obs)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.6, s=30, label="Data")
x_line = np.linspace(-2.5, 2.5, 100)
ax.plot(x_line, true_alpha + true_beta * x_line, "r--", lw=2,
        label=f"True: y = {true_alpha} + {true_beta}x")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Synthetic Data for Bayesian Linear Regression")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Bayesian linear regression in PyMC ---
with pm.Model() as linreg_model:
    # Priors
    alpha = pm.Normal("alpha", mu=0, sigma=10)
    beta = pm.Normal("beta", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=5)

    # Expected value
    mu = alpha + beta * x

    # Likelihood
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)

    # Sample
    trace_linreg = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print(az.summary(trace_linreg, var_names=["alpha", "beta", "sigma"]))

In [ ]:
# --- Plot posterior distributions ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, var, true_val in zip(axes, ["alpha", "beta", "sigma"],
                              [true_alpha, true_beta, true_sigma]):
    samples = az.extract(trace_linreg, var_names=[var])[var].values
    ax.hist(samples, bins=50, density=True, alpha=0.6, color="steelblue",
            edgecolor="white")
    ax.axvline(true_val, color="red", ls="--", lw=2, label=f"True: {true_val}")
    ax.axvline(samples.mean(), color="orange", ls="-", lw=2,
               label=f"Mean: {samples.mean():.2f}")
    ax.set_title(f"Posterior of {var}")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# --- Posterior regression lines ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x, y, alpha=0.5, s=25, zorder=5, label="Data")

# Draw 100 regression lines from the posterior
alpha_post = az.extract(trace_linreg, var_names=["alpha"])["alpha"].values
beta_post = az.extract(trace_linreg, var_names=["beta"])["beta"].values

x_plot = np.linspace(-2.5, 2.5, 100)
for i in rng.choice(len(alpha_post), size=100, replace=False):
    ax.plot(x_plot, alpha_post[i] + beta_post[i] * x_plot,
            color="steelblue", alpha=0.05)

ax.plot(x_plot, true_alpha + true_beta * x_plot, "r--", lw=2.5,
        label="True line")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Posterior Regression Lines")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Credible Intervals vs Confidence Intervals

| | Bayesian Credible Interval | Frequentist Confidence Interval |
|---|---|---|
| **Interpretation** | "There is a 95% probability that $\theta$ lies in this interval" | "If we repeated the experiment many times, 95% of such intervals would contain $\theta$" |
| **Fixed** | Interval bounds are random, $\theta$ is the probability target | $\theta$ is fixed, interval bounds are random |
| **Depends on** | Prior + data | Data only |

Let's demonstrate the difference with a simulation.

In [ ]:
# --- Simulation: frequentist CI coverage vs Bayesian HDI ---
true_mu = 5.0
true_sd = 2.0
sample_size = 30
n_experiments = 200

freq_covers = []
bayes_covers = []
freq_intervals = []
bayes_intervals = []

for i in range(n_experiments):
    data = rng.normal(true_mu, true_sd, size=sample_size)
    sample_mean = data.mean()
    se = data.std(ddof=1) / np.sqrt(sample_size)

    # Frequentist 95% CI (t-distribution)
    t_crit = stats.t.ppf(0.975, df=sample_size - 1)
    ci_low = sample_mean - t_crit * se
    ci_high = sample_mean + t_crit * se
    freq_covers.append(ci_low <= true_mu <= ci_high)
    freq_intervals.append((ci_low, ci_high))

    # Bayesian 95% HDI (Normal prior with mu=0, sigma=10 -- weakly informative)
    prior_mu, prior_sd = 0, 10
    # Posterior of Normal-Normal conjugate with known sigma approximation
    data_var = data.var(ddof=1)
    post_prec = 1 / prior_sd**2 + sample_size / data_var
    post_mu = (prior_mu / prior_sd**2 + sample_size * sample_mean / data_var) / post_prec
    post_sd = np.sqrt(1 / post_prec)

    hdi_low = post_mu - 1.96 * post_sd
    hdi_high = post_mu + 1.96 * post_sd
    bayes_covers.append(hdi_low <= true_mu <= hdi_high)
    bayes_intervals.append((hdi_low, hdi_high))

print(f"Frequentist CI coverage: {np.mean(freq_covers):.1%}")
print(f"Bayesian HDI coverage:   {np.mean(bayes_covers):.1%}")

In [ ]:
# --- Visualize the first 50 intervals ---
n_show = 50

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

for ax, intervals, covers, title in zip(
    axes,
    [freq_intervals[:n_show], bayes_intervals[:n_show]],
    [freq_covers[:n_show], bayes_covers[:n_show]],
    ["Frequentist 95% CI", "Bayesian 95% HDI"]
):
    for i, ((lo, hi), covered) in enumerate(zip(intervals, covers)):
        color = "steelblue" if covered else "red"
        ax.plot([lo, hi], [i, i], color=color, lw=1.5, alpha=0.7)

    ax.axvline(true_mu, color="black", ls="--", lw=1.5, label=f"True $\\mu$ = {true_mu}")
    ax.set_xlabel("Value")
    ax.set_title(title)
    ax.legend(loc="upper right")

axes[0].set_ylabel("Experiment index")
plt.suptitle("Blue = covers true value, Red = misses", fontsize=12)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Bayes' theorem** is a principled rule for updating beliefs with data.
2. **Conjugate models** (e.g., Beta-Binomial) give analytical posteriors -- useful for building intuition.
3. **Prior predictive checks** help us catch unreasonable priors before fitting to data.
4. **PyMC** lets us specify complex models and handles the sampling automatically.
5. **Bayesian linear regression** gives full posterior distributions, not just point estimates.
6. **Credible intervals** have a direct probability interpretation; confidence intervals do not.

---

*Next: Day 2 -- MCMC Methods and Generalized Linear Models*